# Notebook 01 — Chunking Strategy Experiments

Compare clause-boundary chunking vs fixed-token chunking on retrieval quality.

In [ ]:
import sys
sys.path.append('../backend')

from services.ingestion import clause_boundary_chunk, extract_text_from_pdf
import json

# Load a test PDF
PDF_PATH = '../samples/sample_nda.pdf'  # Add your own PDF here
pages = extract_text_from_pdf(PDF_PATH)
print(f'Extracted {len(pages)} pages')

In [ ]:
# Clause-boundary chunks
clause_chunks = clause_boundary_chunk(pages, 'sample_nda.pdf')
print(f'Clause-boundary: {len(clause_chunks)} chunks')
for c in clause_chunks[:5]:
    print(f"  [{c.metadata['clause_type']}] {c.metadata['clause_name'][:60]} — {len(c.text)} chars")

In [ ]:
# Fixed-token chunks (comparison)
from llama_index.core.node_parser import TokenTextSplitter
from llama_index.core import Document

full_text = ' '.join(p['text'] for p in pages)
doc = Document(text=full_text)
splitter = TokenTextSplitter(chunk_size=512, chunk_overlap=50)
token_chunks = splitter.get_nodes_from_documents([doc])

print(f'Fixed-token (512): {len(token_chunks)} chunks')
# Observation: fixed-token chunks often split across clause boundaries,
# mixing content from multiple clauses in one chunk.

In [ ]:
# Visualise clause type distribution
from collections import Counter
import matplotlib.pyplot as plt

types = Counter(c.metadata['clause_type'] for c in clause_chunks)
plt.figure(figsize=(10, 4))
plt.bar(types.keys(), types.values(), color='steelblue')
plt.title('Clause types found in document')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()